# Paperless-ML — demo day runbook

Ordered, copy-pasteable commands for the live demo. Each cell is self-contained so you can skip ahead during the presentation without re-running earlier cells.

## How to use

1. SSH into the freshly provisioned Chameleon node (`ssh cc@<floating-ip>`)
2. `cd ~/paperless-ml`
3. `jupyter lab --ip=0.0.0.0 --port=8888 --no-browser` (or run these cells in a terminal one-by-one if you prefer)
4. Tunnel `ssh -L 8888:localhost:8888 cc@<floating-ip>` from your laptop
5. Open this notebook and work top-to-bottom for warm-up, then jump to Phase 3 cells during the actual demo

## Structure

- **Phase 0** — one-command bring-up on a fresh node (~15 min)
- **Phase 1** — warm-up: Elnath chain + MLflow seed + fixtures (~20 min, automated)
- **Phase 2** — T-30m sanity checks
- **Phase 3** — live demo commands, one cell per rubric bullet
- **Recovery** — reset snippets if something goes sideways mid-demo


---
## Phase 0 — Provisioning

Assumes Chameleon lease active, floating IP attached, SSH key on the node.

In [ ]:
# Sanity — are we on the node and in the right directory?
!hostname && whoami && pwd

In [ ]:
# One-command stack bring-up. ~10-15 min on cold image cache.
# Clones peer repos, creates shared network, starts 17 services,
# extracts admin token, prints service URLs.
!bash scripts/chameleon_setup.sh

In [ ]:
# 13-checkpoint end-to-end test. Must be 13/13 green before Phase 1.
!bash scripts/verify_integration.sh

---
## Phase 1 — Warm-up

Automated: single script handles Elnath's chain (IAM ingest → drift reference → validate_ingestion → synthetic manifest → drift_monitor up → Grafana import), MLflow seed (`paperless-htr` v1+v2, `@production=v2`), fixture upload, rollback-ctrl audit seed, TOKEN export.

All workarounds from `docs/KNOWN_GAPS.md` baked in: sed patch on build_drift_reference.py, inline pip installs for batch_pipeline and drift reference.

Idempotent — safe to re-run after a partial failure.

In [ ]:
!bash scripts/demo_warmup.sh

In [ ]:
# Load the token into this kernel's environment for later cells
import os, pathlib
os.environ['TOKEN'] = pathlib.Path('/tmp/paperless_token').read_text().strip()
print(f"TOKEN length: {len(os.environ['TOKEN'])}")

---
## Phase 2 — T-30m sanity

Run this block 30 minutes before the slot. If any check fails, stop and diagnose — do not try to paper over on-camera.

In [ ]:
# All Prometheus targets UP
%%bash
curl -s http://localhost:9090/api/v1/targets | python3 -c "
import json, sys
for t in sorted(json.load(sys.stdin)['data']['activeTargets'], key=lambda x: x['labels'].get('job','')):
    health = t['health']
    marker = '  OK ' if health == 'up' else '  !! '
    print(f'{marker}{t[\"labels\"].get(\"job\"):22s} {health}')
"

In [ ]:
# MLflow state
%%bash
sg docker -c "docker compose exec -T mlflow python - <<'PY'
from mlflow import MlflowClient
c = MlflowClient('http://localhost:5000')
versions = c.search_model_versions(\"name='paperless-htr'\")
mv = c.get_model_version_by_alias('paperless-htr', 'production')
print(f'  versions: {len(versions)}')
print(f'  @production: v{mv.version}')
PY"
# expect: versions >= 2, @production: v2

In [ ]:
# drift_monitor is exposing metrics + OOD samples staged
%%bash
echo '── drift metrics ──'
curl -s http://localhost:8100/metrics | grep -E '^drift_(checks|events)_total' | sed 's/^/  /'
echo '── OOD samples ──'
sg docker -c 'docker compose exec -T minio mc ls local/paperless-images/ood/' 2>/dev/null | head

In [ ]:
# All services healthy
!sg docker -c 'docker compose ps' | head -25

---
# Phase 3 — Live demo

One cell per rubric bullet, same order as `Demo_script.md`. Each cell is self-contained and executable mid-demo without re-running earlier cells.

Target pacing: 22-25 min for the whole phase; Q&A fills the rest of the 30-min slot.

## Bullet 1 — Bring-up + production data script

Stack is already up. Show `docker compose ps`, then fire the generator so panels populate live.

In [ ]:
# Show the running stack
!sg docker -c 'docker compose ps' | head -20

In [ ]:
# Token was extracted by setup — show it exists
!echo "Token: ${TOKEN:0:8}...${TOKEN: -4}  (length=${#TOKEN})"

In [ ]:
# Fire Elnath's production data generator — runs 2 min in background
!bash scripts/run_data_generator.sh --rate 2.0 --duration 120 &
print('Generator firing in background — switch to Grafana: http://<ip>:3000/d/paperless-ml-overview')

## Bullet 2 — Using the ML feature

Hero command shows semantic merge: total=5 (keyword hit + 4 semantic), ml_semantic_added=4.

In [ ]:
!curl -sf -H "Authorization: Token $TOKEN" "http://localhost:8000/api/search/?query=invoice" | python3 -c "import json,sys; d=json.load(sys.stdin); print(f'total={d[\"total\"]}, ml_semantic_added={d.get(\"ml_semantic_added\",0)}, returned={len(d[\"documents\"])}')"

Browser tabs to open:
- `http://<ip>:8000/` — Paperless UI; search for 'invoice'
- `http://<ip>:8000/ml-ui/doc/<id>/` — HTR regions + confidences on pre-uploaded fixture

## Bullet 3 — Feedback capture

After submitting a correction in the UI and hitting thumbs up/down on search results, verify events hit Redpanda + DB:

In [ ]:
%%bash
# Redpanda tails — use timeout wrapper (rpk --num N blocks waiting for N)
echo '── paperless.corrections tail ──'
sg docker -c "docker compose exec redpanda sh -c 'timeout 3 rpk topic consume paperless.corrections --offset start --format \"%o %v\\n\" 2>/dev/null'" | tail -3
echo '── paperless.feedback tail ──'
sg docker -c "docker compose exec redpanda sh -c 'timeout 3 rpk topic consume paperless.feedback --offset start --format \"%o %v\\n\" 2>/dev/null'" | tail -3

In [ ]:
# DB row count
!sg docker -c "docker compose exec paperless-web python manage.py shell -c 'from ml_hooks.models import Feedback; print(Feedback.objects.count(), \"feedback rows\")'"

## Bullet 4 — Production data saved for retraining

Browser: MinIO console → `paperless-datalake` → `warehouse/` — tour the tree.

Then show the manifest's `data_quality` block in terminal (cleaner than MinIO's download-only UI).

In [ ]:
%%bash
# Find the newest manifest + pretty-print its data_quality block
LATEST=$(sg docker -c 'docker compose exec -T minio mc ls local/paperless-datalake/warehouse/htr_training/' \
  | awk '{print $NF}' | grep '^v_' | sort | tail -1 | tr -d '/\r')
echo "Latest snapshot: $LATEST"
sg docker -c "docker compose exec -T minio mc cat local/paperless-datalake/warehouse/htr_training/${LATEST}/manifest.json" \
  | python3 -c "import json, sys; print(json.dumps(json.load(sys.stdin)['data_quality'], indent=2))"

Narration: *R1-R5 rejection codes, per-reason samples, rubric's data-quality deliverable. One-sentence disclosure: this manifest was seeded — our postgres runs paperless-ngx's schema, not Elnath's corrections tables. Structure matches his `quality.py` line-for-line.*

## Bullet 5 — Retraining + redeployment

Show the scheduler's trigger rules, then bypass them via `force_tick.py` to run one cycle live (~60s).

In [ ]:
# Scheduler running with justified trigger rules
!sg docker -c 'docker compose logs --tail=15 pipeline-scheduler'

In [ ]:
# Force one tick — bypasses thresholds, runs train → eval → gate → promote
!sg docker -c 'docker compose exec pipeline-scheduler python force_tick.py'

In [ ]:
# Verify @production advanced
%%bash
sg docker -c "docker compose exec -T mlflow python - <<'PY'
from mlflow import MlflowClient
c = MlflowClient('http://localhost:5000')
mv = c.get_model_version_by_alias('paperless-htr', 'production')
print(f'@production is now v{mv.version}')
PY"
# Browser: http://<ip>:5050/#/models/paperless-htr

One-sentence caveat to narrate: *The models here are stub-scale — they demonstrate the mechanism, not a production TrOCR fine-tune. The pipeline, gate, registry, MinIO export, and ml-gateway restart are all real.*

## Bullet 6 — Safeguarding plan + automated rollback

Tour `docs/SAFEGUARDING.md`, then fire the three-path rollback demo.

In [ ]:
# Case 1 — Happy path: version swap + ml-gateway restart
%%bash
sg docker -c "docker compose exec rollback-ctrl sh -c '
python - <<PY
import json, urllib.request
payload = {\"alerts\":[{\"status\":\"firing\",\"labels\":{
    \"alertname\":\"HTRCorrectionRateHigh\",
    \"rollback_trigger\":\"true\",
    \"severity\":\"critical\"
}}]}
req = urllib.request.Request(\"http://localhost:8000/webhook\",
    data=json.dumps(payload).encode(), headers={\"Content-Type\":\"application/json\"})
print(urllib.request.urlopen(req).read().decode())
PY'"

In [ ]:
# Case 2 — Cooldown: re-fire same alertname within 5 min
%%bash
sg docker -c "docker compose exec rollback-ctrl sh -c '
python - <<PY
import json, urllib.request
payload = {\"alerts\":[{\"status\":\"firing\",\"labels\":{
    \"alertname\":\"HTRCorrectionRateHigh\",
    \"rollback_trigger\":\"true\",
    \"severity\":\"critical\"
}}]}
req = urllib.request.Request(\"http://localhost:8000/webhook\",
    data=json.dumps(payload).encode(), headers={\"Content-Type\":\"application/json\"})
print(urllib.request.urlopen(req).read().decode())
PY'"

In [ ]:
# Case 3 — Version floor: fire different alertname until @production is v1
%%bash
sg docker -c "docker compose exec rollback-ctrl sh -c '
python - <<PY
import json, urllib.request
payload = {\"alerts\":[{\"status\":\"firing\",\"labels\":{
    \"alertname\":\"SearchCTRLow\",
    \"rollback_trigger\":\"true\",
    \"severity\":\"critical\"
}}]}
req = urllib.request.Request(\"http://localhost:8000/webhook\",
    data=json.dumps(payload).encode(), headers={\"Content-Type\":\"application/json\"})
print(urllib.request.urlopen(req).read().decode())
PY'"

In [ ]:
# Tail the audit log in a second pane during the 3 cases
!sg docker -c 'docker compose logs --tail=20 rollback-ctrl' | grep -E 'rollback-trigger|ROLLBACK'

## Bullet 7 — Per-role monitoring

### 7a — Serving (Yikai)

Grafana → `paperless-ml-overview` (9 panels). Prometheus → `/alerts` (8 rules). Rollback audit trail.

In [ ]:
%%bash
# Rule groups + counts
curl -s http://localhost:9090/api/v1/rules | python3 -c "
import json, sys
for g in json.load(sys.stdin)['data']['groups']:
    print(f'  {g[\"name\"]}: {len(g[\"rules\"])} rules')
"
echo '── active scrape targets ──'
curl -s http://localhost:9090/api/v1/targets | python3 -c "
import json, sys
for t in sorted(json.load(sys.stdin)['data']['activeTargets'], key=lambda x: x['labels'].get('job','')):
    print(f'  {t[\"labels\"].get(\"job\"):22s} {t[\"health\"]}')
"

### 7b — Training (Dongting)

MLflow → three experiments (training, evaluation, quality_gate). Model registry with gate-passing versions.

In [ ]:
# Registry queryable via API
!curl -s "http://localhost:5050/api/2.0/mlflow/registered-models/search?max_results=10" | python3 -m json.tool | head -30

### 7c — Data (Elnath) — three points

**Point 3 (live drift)**: Grafana drift dashboard, then inject OOD crops.

In [ ]:
%%bash
# Drift metrics baseline
curl -s http://localhost:8100/metrics | grep -E '^drift_(checks|events)_total' | sed 's/^/  /'

In [ ]:
%%bash
# Push OOD crops — self-healing enumeration from MinIO
for key in $(sg docker -c 'docker compose exec -T minio mc ls local/paperless-images/ood/' | awk '{print $NF}' | tr -d '\r'); do
  curl -sS -X POST http://localhost:8100/drift/check \
    -H 'Content-Type: application/json' \
    -d "{\"crop_s3_url\":\"s3://paperless-images/ood/${key}\"}"
  echo
done
echo '── after injection ──'
curl -s http://localhost:8100/metrics | grep -E '^drift_(checks|events)_total' | sed 's/^/  /'

**Point 1 (ingestion validation)**: MinIO console → `warehouse/iam_dataset/_validation/<ts>.json` and `warehouse/squad_dataset/_validation/<ts>.json`, point at `checks` array.

In [ ]:
%%bash
# Terminal fallback — show IAM validation checks block
for ds in iam_dataset squad_dataset; do
  LATEST=$(sg docker -c "docker compose exec -T minio mc ls local/paperless-datalake/warehouse/${ds}/_validation/" \
    | awk '{print $NF}' | tr -d '\r' | sort | tail -1)
  echo "── ${ds}: ${LATEST} ──"
  sg docker -c "docker compose exec -T minio mc cat local/paperless-datalake/warehouse/${ds}/_validation/${LATEST}" 2>/dev/null \
    | python3 -c "import json, sys; d=json.load(sys.stdin); print(json.dumps({'checks': d.get('checks', [])}, indent=2))" 2>/dev/null | head -25
done

**Point 2 (training-set quality)**: already shown in Bullet 4 manifest — re-attribute here.

## Bullet 8 — Bonus items (narration only)

- **Serving (Yikai)**: Triton init-impl bonus; replaced by FastAPI+ONNX Runtime for system impl. Code preserved in `serving/src/export/`.
- **Training (Dongting)**: Triton-ready packaging at `outputs/triton_repo/`; feeds into `deploy_model.sh` as source format.
- **Data (Elnath)**: online drift monitor (MMD); fully integrated, seen in bullet 7c point 3.

## Closing — repo URLs

```
  Palomarr/paperless-ml
  REDES01/paperless_data
  REDES01/paperless_data_integration
  gdtmax/paperless_training_integration
```

One-command bring-up: `bash scripts/chameleon_setup.sh`

---
# Recovery snippets

If something breaks mid-demo — don't panic, run the relevant cell.

In [ ]:
# Reset MLflow @production to v2 (after a dry-run or accidental rollback)
%%bash
sg docker -c "docker compose exec -T mlflow python - <<'PY'
from mlflow import MlflowClient
c = MlflowClient('http://localhost:5000')
c.set_registered_model_alias('paperless-htr', 'production', '2')
mv = c.get_model_version_by_alias('paperless-htr', 'production')
print(f'@production reset to v{mv.version}')
PY"

In [ ]:
# ml-gateway hung after restart — kick it
!sg docker -c 'docker compose restart ml-gateway' && echo 'Restart triggered — ~90s cold boot for TrOCR+mpnet'

In [ ]:
# Drift-monitor silent? Check it's still on paperless_ml_net
!sg docker -c "docker inspect drift_monitor --format '{{range \$k,\$v := .NetworkSettings.Networks}}{{\$k}} {{end}}'"

In [ ]:
# Prometheus lost a target — hot-reload config
!curl -X POST http://localhost:9090/-/reload && echo 'Reloaded'

In [ ]:
# Token expired or shell lost it — re-extract + re-export in this kernel
import os, subprocess
os.environ['TOKEN'] = subprocess.check_output(
    ['bash', 'scripts/get_paperless_token.sh'], text=True).strip()
print(f"Token length: {len(os.environ['TOKEN'])}")